In [1]:
#In cloud server, a model is trained

In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import sklearn
import sklearn.datasets
import sklearn.ensemble
import lime
import lime.lime_tabular
from __future__ import print_function

# System Call Log Creation:

In [3]:
import csv
# Write header
#writer.writerow(["Itr_Num", "Layer_Name", "Process_Name", "Log_Prompt"])
global Track_log
Track_log=1
def get_last_track_log(csv_filename):
    try:
        with open(csv_filename, mode="r", newline="") as file:
            reader = csv.reader(file)
            rows = list(reader)
            if rows:  # Ensure the file is not empty
                last_row = rows[-1]  # Get the last row
                return (last_row[-1])  # Extract the last column (Track_log)
            else:
                return 1  # Default value if file is empty
    except FileNotFoundError:
        return 1  # Default if file doesn't exist


def Call_log( Layer_Name, Process_Name, Log_Prompt):
    # Define the CSV file name
    csv_filename = "System_Call_Log.csv"
    # Write data to CSV file
    global Track_log
    Track_log=get_last_track_log(csv_filename)
    itr_num=Track_log
    with open(csv_filename, mode="a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([itr_num, Layer_Name, Process_Name, Log_Prompt,Track_log])

# Task 1.Model Training
# step 1:data reading and preprocessing

In [4]:
df=pd.read_csv('./Soil_Temp_measure.csv', low_memory=False) 
#df.info()

In [5]:
##Data cleaning
from sklearn.utils import shuffle
drop_columns = ["Unnamed: 20","DateTime"]

df.drop(drop_columns, axis=1, inplace=True)
df.dropna(axis=0, how='any', inplace=True)
df.drop_duplicates(subset=None, keep="first", inplace=True)
df = shuffle(df)
df.isna().sum()

Soil temperture at 2 inch below the soil surface (F)          0
Soil data under the cover right on top of soil surface (F)    0
Air temperature 1.5m (F)                                      0
Dewpoint                                                      0
Wind Chill                                                    0
Wind Gust 10m                                                 0
Wind Speed 10m                                                0
Wind Direction 10m                                            0
Pressure (mb)                                                 0
Pressure (inHg)                                               0
Rainfall                                                      0
Relative Humidity                                             0
Approximate Max                                               0
Solar Radiation                                               0
Snow cover                                                    0
Turfgrass species                       

In [6]:
#To replace white spaces with other characters (underscore for instance):
df.columns = df.columns.str.strip()#To remove white space at both ends:
df.columns = df.columns.str.replace(' ', '_')
df[['Cover_or_not']] = df[['Cover_or_not']].replace(' ', '', regex=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 427 entries, 431 to 333
Data columns (total 19 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Soil_temperture_at_2_inch_below_the_soil_surface_(F)        427 non-null    float64
 1   Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)  427 non-null    float64
 2   Air_temperature_1.5m_(F)                                    427 non-null    float64
 3   Dewpoint                                                    427 non-null    float64
 4   Wind_Chill                                                  427 non-null    float64
 5   Wind_Gust_10m                                               427 non-null    float64
 6   Wind_Speed_10m                                              427 non-null    float64
 7   Wind_Direction_10m                                          427 non-null    float64
 8 

In [7]:
# Use set() to eliminate duplicate values in column 'Snow cover'
unique_values_set_cover = set(df["Cover_or_not"])
# Print the unique values
#print(unique_values_set_cover)

In [8]:
#Data preprocessing and saving to another file
df.to_csv('preprocessed_soil_temp_stillwater.csv', encoding='utf-8', index=False)
df = pd.read_csv('./preprocessed_soil_temp_stillwater.csv', low_memory=False) 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 427 entries, 0 to 426
Data columns (total 19 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Soil_temperture_at_2_inch_below_the_soil_surface_(F)        427 non-null    float64
 1   Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)  427 non-null    float64
 2   Air_temperature_1.5m_(F)                                    427 non-null    float64
 3   Dewpoint                                                    427 non-null    float64
 4   Wind_Chill                                                  427 non-null    float64
 5   Wind_Gust_10m                                               427 non-null    float64
 6   Wind_Speed_10m                                              427 non-null    float64
 7   Wind_Direction_10m                                          427 non-null    float64
 8   

# Step 2: data training

In [9]:
#Dividing data into X and Y
feat_cols = list(df.columns)
label_col = "Cover_or_not"
feat_cols.remove(label_col)
X = df.drop([label_col], axis=1)
y = df[label_col]

feat_cols = list(X.columns)
from sklearn.preprocessing import LabelEncoder
categorical_index=[]
feat_cols = list(X.columns)
categorical_names = {}
##label encoding to object values#######################################################
categorical_features = ["Snow_cover",'Turfgrass_species','Mowing_height','Location']
categorical_index = [feat_cols.index(i) for i in categorical_features]
#print(categorical_index)
# for i in categorical_features:
#     z=feat_cols.index(i)
#     print(z)
#     categorical_index.append(z)
df.info()
#print("feat_col_len"+str(len(feat_cols)))
def Label_encoding(X, name,index):
    le = LabelEncoder()
    label = le.fit_transform(X[name])
   # df.drop(name, axis=1, inplace=True)
    X[name] = label
    le_value_mapping = dict(zip(le.transform(le.classes_), le.classes_))
    categorical_names[index] = le_value_mapping
    #categorical_names[index] = le.classes_
    #print(name)
    #print(le.classes_)

for i in categorical_features:
    z=feat_cols.index(i)
    Label_encoding(X,i,z)
#print("after encoding \n")
X.info()


categorical_names
# feat_cols = list(df.columns)
# feat_cols.remove(label_col)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 427 entries, 0 to 426
Data columns (total 19 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Soil_temperture_at_2_inch_below_the_soil_surface_(F)        427 non-null    float64
 1   Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)  427 non-null    float64
 2   Air_temperature_1.5m_(F)                                    427 non-null    float64
 3   Dewpoint                                                    427 non-null    float64
 4   Wind_Chill                                                  427 non-null    float64
 5   Wind_Gust_10m                                               427 non-null    float64
 6   Wind_Speed_10m                                              427 non-null    float64
 7   Wind_Direction_10m                                          427 non-null    float64
 8   

{14: {0: 'No', 1: 'Yes'},
 15: {0: 'Bermudagrass'},
 16: {0: '0.125"'},
 17: {0: 'Stilwlater, OK'}}

In [10]:
#############################splitting data test train data############################
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

#del X
#del y

from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()
y_train =  label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

In [11]:
label_encoder.classes_

array(['No', 'Yes'], dtype=object)

In [12]:
#Build the model with Random Forest Regressor :
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(max_depth=6, random_state=0, n_estimators=10)
model.fit(X_train, y_train)

RandomForestRegressor(max_depth=6, n_estimators=10, random_state=0)

# Step3: save the model to disk

In [13]:
# save the model to disk
import pickle
filename = 'snowcover_model.sav'
pickle.dump(model, open(filename, 'wb'))

# Step 4: Testing Model prediction

In [14]:
y_pred = model.predict(X_test)
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_pred)**(0.5)
y_pred

array([0. , 0. , 0. , 1. , 0. , 0. , 0. , 1. , 0.1, 1. , 0.5, 0. , 0.1,
       0. , 0. , 0. , 0. , 0.2, 1. , 0. , 0. , 0. , 0. , 0. , 1. , 1. ,
       0. , 1. , 0.1, 0. , 0. , 0. , 0. , 1. , 0. , 0.2, 0. , 0. , 1. ,
       0. , 0. , 1. , 0. , 0. , 0. , 0. , 1. , 0. , 0.5, 0. , 0. , 1. ,
       0. , 0. , 0.2, 0. , 1. , 1. , 0. , 1. , 0. , 0. , 1. , 0.1, 0. ,
       1. , 1. , 1. , 0. , 0. , 1. , 0. , 0. , 0. , 1. , 0. , 0. , 0. ,
       0. , 0. , 0. , 1. , 0. , 1. , 1. , 0. ])

In [15]:
print('Random Forest MSError', np.mean((model.predict(X_test) - y_test) ** 2))
#from sklearn.metrics import r2_score
model.score(X_test, y_test)
print('MSError when predicting the mean', np.mean((y_train.mean() - y_test) ** 2))


Random Forest MSError 0.014651162790697676
MSError when predicting the mean 0.21991304944337928


# Task 2:Process Flow in Cloud Server

In [16]:
def Label_encoding(X, name,index):
    le = LabelEncoder()
    label = le.fit_transform(X[name])
   # df.drop(name, axis=1, inplace=True)
    X[name] = label
    le_value_mapping = dict(zip(le.transform(le.classes_), le.classes_))
    categorical_names[index] = le_value_mapping
    #categorical_names[index] = le.classes_
    #print(name)
    #print(le.classes_)
    return X

In [17]:
def PreprocessedSensorData(X):
    #To replace white spaces with other characters (underscore for instance):
    X.columns = X.columns.str.strip()#To remove white space at both ends:
    X.columns = X.columns.str.replace(' ', '_')
    
    feat_cols = list(X.columns)
    from sklearn.preprocessing import LabelEncoder
    categorical_index=[]
    feat_cols = list(X.columns)
    categorical_names = {}
    ##label encoding to object values#######################################################
    categorical_features = ["Snow_cover",'Turfgrass_species','Mowing_height','Location']
    categorical_index = [feat_cols.index(i) for i in categorical_features]
    #print(categorical_index)
    for i in categorical_features:
        z=feat_cols.index(i)
        cleaned_X=Label_encoding(X,i,z)
    #print("after encoding \n")
    #cleaned_X.info()
    return cleaned_X
    

In [18]:
import pickle
def ExecuteMLModel():
    print("-----------------------Loaded model-------------------------")
    # load the model from disk
    filename= 'snowcover_model.sav'
    loaded_model = pickle.load(open(filename, 'rb'))
    return loaded_model
    

In [19]:

def GetMLModelPrediction(loaded_model,X):
    
    #Retrieve list of training features names from classifier
    cols_when_model_builds = loaded_model.feature_names_in_
    print("cols_when_model_builds",cols_when_model_builds)
    X = X[cols_when_model_builds]
    y=loaded_model.predict(X)
    print("Model prediction for y",y)
    return y

In [20]:
import numpy as np
##0 means no covering needed
##1 means covering needed
triggerActuator=0
def DecidetoTriggerActuator(y):
    global triggerActuator
    print("-----------------------Loaded model-------------------------")
    result=np.bincount(y.astype(int)).argmax()
    print("y.mode,triggerActuator",result,triggerActuator)
    Call_log( "Cloud Layer", "DecidetoTriggerActuator", "DecidetoTriggerActuator")
    if result!=triggerActuator:
        triggerActuator=int(result)
        ShowAnalysisResult_application(triggerActuator)
         #Send the processed data back to the edge
        hostname = socket.gethostname()
        #edge_ip = '192.168.137.181'#socket.gethostbyname(hostname)
        edge_ip='192.168.0.20'
        message="triggerActuator "+ str(triggerActuator)
        SendTriggerToActuator(message, edge_ip, 12347)  # Adjust IP and port to the edge server's listening address
        #send_data_to_edge

        
    

In [21]:
import requests
def ShowAnalysisResult_application(modelprediction):
    # Simulate some work done by the client
    
    print("modelprediction",modelprediction)
    if modelprediction==0:
        output='No Covering Needed'
    else:
        output="Yes Cover Please"
    result = {
        "task": "Soil Cover",
        "status": "Completed",
        "output": output 
    }

    # Send the result to the server
    response = requests.post('http://localhost:5000/result', json=result)

    # Check the server's response
    print(response.json())

    

In [22]:
# import pandas as pd
# import numpy as np
# from matplotlib import pyplot as plt
# import seaborn as sns
# import sklearn
# import sklearn.datasets
# import sklearn.ensemble
# import lime
# import lime.lime_tabular
# from __future__ import print_function
# df=pd.read_csv('./Soil_Temp_measure.csv', low_memory=False) 
# ##Data cleaning
# from sklearn.utils import shuffle
# drop_columns = ["Unnamed: 20","DateTime"]

# df.drop(drop_columns, axis=1, inplace=True)
# df.dropna(axis=0, how='any', inplace=True)
# df.drop_duplicates(subset=None, keep="first", inplace=True)
# df = shuffle(df)
# df.isna().sum()

# # mm=mm = df.groupby(['Snow cover', 'Turfgrass species', 'Mowing height', 'Location']).mean().reset_index()

# # mm.info()

# # modelprediction=model_prediction(datacleaning(mm))
# # Send_To_Application(modelprediction)
# # compare_results(y)

In [23]:
import socket
import json
def ReceivedSensorData(ip, port):#ReceivedSensorData
    # Create a TCP/IP socket
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    # Bind the socket to the address and port
    server_socket.bind((ip, port))

    # Listen for incoming connections
    server_socket.listen(5)
    print(f"TCP server listening on {ip}:{port}")

    while True:
        # Wait for a connection
        client_socket, client_address = server_socket.accept()
        print(f"Connection from {client_address} established.")

        try:
            # Receive the data in small chunks
            data = b""
            while True:
                chunk = client_socket.recv(1024)
                if not chunk:
                    break
                data += chunk

            # Decode and process the received data
            mean_data = data.decode('utf-8')
#             
            print("Received mean DataFrame:")
#             print(mean_data)
        
            # Convert string to dictionary
            data_dict = json.loads(mean_data)
            # Convert dictionary to pandas DataFrame
            SensorData = pd.DataFrame(data_dict)
            Call_log( "Cloud Layer", "ReceivedSensorData", "ReceivedSensorData from Edge layer")
            #SensorData.info()
            #print("SensorData",SensorData)
            #FeedSensorDataToMLModel
            FeedSensorDataToMLModel=PreprocessedSensorData(SensorData)
            Call_log( "Cloud Layer", "FeedSensorDataToMLModel", "SensorData has been fed to ML model")
            #ExecuteMLModel
            MLModel=ExecuteMLModel()
            Call_log( "Cloud Layer", "ExecuteMLModel", "MLModel has been executed")
            #GetMLModelPrediction
            modelprediction=GetMLModelPrediction(MLModel,FeedSensorDataToMLModel)#PreprocessedSensorData
            Call_log( "Cloud Layer", "GetMLModelPrediction", "GetMLModelPrediction")
            #Send_To_Application(modelprediction)
            DecidetoTriggerActuator(modelprediction)
           
           
            


        except Exception as e:
            print(f"Error handling client {client_address}: {e}")


        finally:
            # Close the connection
            client_socket.close()

# # Run the TCP server on localhost at port 12346
# host = socket.gethostname()
# tcp_server(host, 12346)

# # Run the TCP server on localhost at port 12346
# # Example IP and port of the original UDP server
# udp_server_ip = '127.0.0.1'
# udp_server_port = 12346
# host = socket.gethostname()
# tcp_server(host, 12356, host, udp_server_port)

In [24]:
# Function to send data back to the edge server
def SendTriggerToActuator(data, edge_ip, edge_port):
    # Create a TCP socket to send data to the edge server
    tcp_client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    print(f"Sending processed data to edge server at {edge_ip}:{edge_port}")
    
    try:
        # Connect to the edge server
        tcp_client.connect((edge_ip, edge_port))
        
        # Send the processed data
        tcp_client.sendall(data.encode('utf-8'))
        Call_log( "Cloud Layer", "SendTriggerToActuator", "Trigger actuator has been sent to Edge layer")
        print(f"Processed data sent to edge server: {data}")
    except Exception as e:
        print(f"Error sending data to edge server: {e}")
    finally:
        tcp_client.close()


In [ ]:
import socket
import threading
# Main function to start the cloud server
def main():
    # Set the IP and port for receiving data from the edge
    hostname = socket.gethostname()
    #tcp_ip = socket.gethostbyname(hostname)
    tcp_ip='0.0.0.0'
    tcp_receive_port = 12346  # Port to receive data from edge server

    # Start the cloud server to listen for data from the edge
    receive_thread = threading.Thread(target=ReceivedSensorData, args=(tcp_ip, tcp_receive_port), daemon=True)
    receive_thread.start()

    # Keep the main thread alive
    receive_thread.join()

if __name__ == '__main__':
    main()

TCP server listening on 0.0.0.0:12346
Connection from ('192.168.0.20', 54651) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Model prediction for y [1.]
-----------------------Loaded model-------------------------
y.mode,triggerActuator 1 0
modelprediction 1
{'data': {'output': 'Yes Cover Please', 'status': 'Completed', 'task': 'Soil Cover'}, 'status': 'Received'}
Sending processed data to edge server at 192.168.0.20:12347
Processed data sent to edge server: triggerActuator 1
Connection from ('192.168.0.20', 54668) established.
Re

Connection from ('192.168.0.20', 54695) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Model prediction for y [1. 0.]
-----------------------Loaded model-------------------------
y.mode,triggerActuator 0 0
Connection from ('192.168.0.20', 54696) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill

Connection from ('192.168.0.20', 54785) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Model prediction for y [0.2]
-----------------------Loaded model-------------------------
y.mode,triggerActuator 0 1
modelprediction 0
{'data': {'output': 'No Covering Needed', 'status': 'Completed', 'task': 'Soil Cover'}, 'status': 'Received'}
Sending processed data to edge server at 192.168.0.20:12347
Processed data sent to edge server: triggerActuator 0
Connection from ('192.168.0.20', 54789) established.
Received mean DataFrame:
------------

Connection from ('192.168.0.20', 54831) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Model prediction for y [0. 1.]
-----------------------Loaded model-------------------------
y.mode,triggerActuator 0 0
Connection from ('192.168.0.20', 54833) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill

Connection from ('192.168.0.20', 54859) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Model prediction for y [0.1]
-----------------------Loaded model-------------------------
y.mode,triggerActuator 0 0
Connection from ('192.168.0.20', 54861) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 

...

-----------------------Loaded model-------------------------
y.mode,triggerActuator 1 0
modelprediction 1
{'data': {'output': 'Yes Cover Please', 'status': 'Completed', 'task': 'Soil Cover'}, 'status': 'Received'}
Sending processed data to edge server at 192.168.0.20:12347
Processed data sent to edge server: triggerActuator 1
Connection from ('192.168.0.20', 55680) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Model prediction for y [0. 0.]
-----------------------Loaded model-------------------------
y.mode,triggerActuator 0 1
mo

Connection from ('192.168.0.20', 55758) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Model prediction for y [0.]
-----------------------Loaded model-------------------------
y.mode,triggerActuator 0 0
Connection from ('192.168.0.20', 55766) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' '

Connection from ('192.168.0.20', 55821) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Error handling client ('192.168.0.20', 55821): "['Wind_Chill'] not in index"
Connection from ('192.168.0.20', 55826) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_

Connection from ('192.168.0.20', 56072) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Model prediction for y [0.]
-----------------------Loaded model-------------------------
y.mode,triggerActuator 0 1
modelprediction 0
{'data': {'output': 'No Covering Needed', 'status': 'Completed', 'task': 'Soil Cover'}, 'status': 'Received'}
Sending processed data to edge server at 192.168.0.20:12347
Processed data sent to edge server: triggerActuator 0
Connection from ('192.168.0.20', 56082) established.
Received mean DataFrame:
-------------

Connection from ('192.168.0.20', 56215) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_Direction_10m' 'Pressure_(mb)' 'Pressure_(inHg)'
 'Rainfall' 'Relative_Humidity' 'Approximate_Max' 'Solar_Radiation'
 'Snow_cover' 'Turfgrass_species' 'Mowing_height' 'Location']
Error handling client ('192.168.0.20', 56215): "['Wind_Chill'] not in index"
Connection from ('192.168.0.20', 56219) established.
Received mean DataFrame:
-----------------------Loaded model-------------------------
cols_when_model_builds ['Soil_temperture_at_2_inch_below_the_soil_surface_(F)'
 'Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)'
 'Air_temperature_1.5m_(F)' 'Dewpoint' 'Wind_Chill' 'Wind_Gust_10m'
 'Wind_Speed_10m' 'Wind_